# Session 21 · LangChain — your first build
## BankAssist, written by you

**Today is different.** Until now you ran notebooks someone else wrote. Today you write the code yourself, in your
own file, in your own repository: `s21/bankassist_lc.py`. This notebook is your guide and your checker. It explains
each step in detail, shows you what to type, and after every step a check cell runs **your** code and tells you
whether it works.

| Part | What it is |
|---|---|
| **Part A · Read before you build** | Why a framework, the words you need, how today works, the rules for using ChatGPT |
| **Part B · Build** | Section 1 sets up your file. Sections 2–6 are the five TODOs, one per section. Section 7 breaks it on purpose. Sections 8–9 check and submit. |
| **Part C · Wrap-up** | Summary, honest limits, interview questions, homework |

**Where you type.** Not in this notebook. Open `s21/bankassist_lc.py` in VS Code (Section 1 creates it), type each
TODO there, save, then come back here and run that section's check cell. You can also run your file directly from a
terminal in the course repository folder:
```
uv run python $env:WORK_REPO\s21\bankassist_lc.py
```

---
# Part A · Read before you build

## A1. Why this session matters

In Session 20 you saw the full round trip of tool calling, raw: a hand-written JSON schema, `json.loads` on the
arguments, a hand-made `tool_call_id`, a loop. It worked — and it was about a hundred lines of careful plumbing.
Every team that builds LLM applications writes that same plumbing, so frameworks exist to write it for them.
**LangChain** is the most widely used one.

A framework doesn't give the model new powers. BankAssist today does nothing it couldn't do in Session 20. What
changes is how much code you write, and how much is hidden from you. Think of an automatic gearbox: same engine,
same road, fewer things to do by hand. And because you drove the manual in Session 20, you'll recognise every hidden
gear change.

**What you learn today, and why it matters at work:**

| You learn to... | It saves you from... |
|---|---|
| Build a chat model, a prompt template and a chain | Rewriting client code for every provider and prompt |
| Turn a Python function into a tool with `@tool` | Hand-writing and maintaining JSON schemas |
| Write the tool loop yourself with LangChain's pieces | Debugging a framework loop you never understood |
| Pin versions and brief your AI assistant | Code from old tutorials (or ChatGPT) that won't run |
| Read framework output and name the S20 step behind it | Being stuck when the framework does something surprising |

**Why interviewers care.** "Have you used LangChain?" is common. The strong answer isn't "yes" — it's "yes, and here's
what it does under the hood, and here's when I'd skip it." After today you can give that answer.

## A2. Where this session fits
```
 S14  system prompt as a contract ─────► SYSTEM in your file
 S16  structured prompts, templates ───► ChatPromptTemplate
 S19  the messages list ───────────────► SystemMessage / HumanMessage / AIMessage / ToolMessage
 S20  tools, raw ──────────────────────► @tool, bind_tools, tool_calls, ToolMessage — same protocol, less code
  │
 S21  LangChain — YOU build BankAssist ► make_llm · make_chain · calculate_emi · ask
  │
 S22  LiteLLM ─────────────────────────► one call format for many providers, fallbacks, cost, a gateway
 S23  FastAPI + SSE ───────────────────► serve BankAssist, streamed
 Module 5  RAG ────────────────────────► retrievers plug into the same chains
 Module 6  Agents ─────────────────────► your loop, prebuilt — now you know what it hides
```

## A3. The words you need today
The full table is in `module03/terms_s21.md`. The six you'll use in the next hour:

- **`ChatOpenAI`** — the model as one Python object. Works with Groq and Ollama because your `.env` sets `BASE_URL`.
- **Messages** — `SystemMessage`, `HumanMessage`, `AIMessage`, `ToolMessage`: Session 19's list, with types.
- **The pipe `|` (LCEL)** — `prompt | llm | parser`: the output of the left becomes the input of the right.
- **`@tool`** — turns a function into a tool. **The docstring becomes the description, so the docstring is your prompt.**
- **`bind_tools`** — attaches the tools to the model once.
- **`tool_calls`** — on the `AIMessage`, a list of dicts (`name`, `args`, `id`). `args` is already a dict.

## A4. How today works — and the rules for ChatGPT

```
 course repo (you pull, never edit)          YOUR work repo (you write, commit, push)
 ┌───────────────────────────────┐           ┌─────────────────────────────────┐
 │ module03/s21_starter.py ──────┼── copy ──►│ s21/bankassist_lc.py  ← YOU TYPE │
 │ module03/s21_check.py         │           └───────────────┬─────────────────┘
 │ module03/s21_version_card.md  │                           │ imported by
 │ module03/s21_common_errors.md │◄──────────────────────────┘ the check cells
 │ this notebook                 │
 └───────────────────────────────┘
```

**Five TODOs, in order:** (1) the model, (2) the chain, (3) the tool, (4) the loop, (5) a second tool — optional.
After each one, run your file or the check cell. Don't type all five and then run — find each mistake while it's small.

**Using ChatGPT is allowed. Three rules:**
1. **Paste `module03/s21_version_card.md` first.** Without it, ChatGPT writes LangChain code from old tutorials
   (`LLMChain`, `ConversationBufferMemory`, `initialize_agent`) that won't run with today's versions.
2. **Never paste your `.env` or your API key.** Not even to "check it".
3. **Ask it to explain or fix your code, not to write the file.** At the exit check, three people explain one
   function each, in their own words.

**Stuck?** Read the last line of the error, then look it up in `module03/s21_common_errors.md`.

---
# Part B · Build

## Section 1 · Set up your file
**What and why.** You need your own copy of the starter inside your work repository, so the code you write is yours,
committed under your name, and on your GitHub profile.

**What the next cell does.** 1) Finds the course repository root (Jupyter may start inside `module03/`). 2) Loads
your `.env`. 3) Copies `s21_starter.py` to `<WORK_REPO>/s21/bankassist_lc.py` — **only if it isn't there already**, so
re-running this cell never wipes your work. 4) Prints the path to open in VS Code.

Before running it: `git pull` and `uv sync` in the course repository (two new packages today: `langchain` and
`langchain-openai`, pinned to exact versions), and make sure `WORK_REPO` is set in `.env`.

In [ ]:
import os, sys, shutil
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "module02").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "module02"))
sys.path.insert(0, str(ROOT / "module03"))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

WORK_REPO = Path(os.getenv("WORK_REPO") or r"C:\Users\you\Documents\genai-course-work")   # <-- your folder
target = WORK_REPO / "s21" / "bankassist_lc.py"
if target.exists():
    print("Already there - NOT overwritten (your work is safe):", target)
else:
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(ROOT / "module03" / "s21_starter.py", target)
    print("Created:", target)
print()
print("Now open it in VS Code:   code", target)
print("Model:", os.getenv("MODEL"), "  endpoint:", os.getenv("BASE_URL"))

from s21_check import check_s21_work, load_student
def show(step):
    """Run the checker on YOUR file and print only the lines for one TODO."""
    for r in check_s21_work(WORK_REPO):
        if step in r.name or (r.name.startswith(("work repo", "s21/", "your file")) and not r.passed):
            print(("PASS  " if r.passed else "FAIL  ") + r.name + (f"\n      -> {r.detail}" if r.detail else ""))

**What you should see.** `Created: ...\s21\bankassist_lc.py` (or `Already there`), the `code ...` line, and your
model. Run the `code ...` line in a terminal, or open the file from VS Code's Explorer.

**Check yourself.** Run your file once now, untouched:
`uv run python $env:WORK_REPO\s21\bankassist_lc.py`. It should print `STEP 1: not done yet (TODO 1)` and
`STEP 3: not done yet (TODO 3)`. That's the starting line, not an error.

## Section 2 · TODO 1 — one model, one call
**What and why.** Everything starts with a model object. In Session 20 you made a client and called
`client.chat.completions.create(...)` every time, then dug the text out of `r.choices[0].message.content`. LangChain
wraps all of that in one object.

**The idea.** `ChatOpenAI` speaks the OpenAI API format. Groq and Ollama both speak it too, so the same object works
with either. Your `.env` decides which, through `BASE_URL`. `temperature=0` keeps answers repeatable; `max_tokens=1500`
leaves room for reasoning models such as gpt-oss.

**What to type** — in `make_llm()`, replace the `raise NotImplementedError("TODO 1")` line with:
```python
return ChatOpenAI(model=os.environ["MODEL"], base_url=os.environ["BASE_URL"],
                  api_key=os.environ["API_KEY"], temperature=0, max_tokens=1500)
```
Save the file, then run the next cell.

In [ ]:
show("TODO 1")
mod = load_student(WORK_REPO)
try:
    reply = mod.make_llm().invoke("Say hello in five words.")
    print()
    print("type:", type(reply).__name__)
    print("text:", reply.content)
except NotImplementedError as e:
    print("not done yet:", e)

**What you should see.** `PASS TODO 1`, `type: AIMessage`, and a short greeting.

**Find the S20 piece that vanished.** `reply.content` replaces `r.choices[0].message.content`. The reply is a typed
`AIMessage` — the same kind of object your loop will receive in TODO 4.

**Check yourself.** Why does the same code work on Groq and on Ollama? (Both speak the OpenAI API; only `BASE_URL`
and `MODEL` change, and they come from `.env`.) *Common mistake:* typing your key into the code instead of reading it
from `os.environ` — then it ends up on GitHub.

## Section 3 · TODO 2 — prompt, model, parser: your first chain
**What and why.** BankAssist needs its system prompt on every call, plus the customer's question. In Session 16 you
built prompts with templates and f-strings, and parsed the reply yourself. LangChain makes each of those a piece, and
the pipe `|` connects them.

**The idea.**
```
 {"question": "..."} ──► ChatPromptTemplate ──► ChatOpenAI ──► StrOutputParser ──► "plain text"
                         (fills SYSTEM +        (one call)     (pulls out .content)
                          {question})
```
Read `prompt | llm | parser` left to right as an assembly line: each piece's output is the next piece's input.

**What to type** — in `make_chain(llm)`, replace the `raise` line with:
```python
prompt = ChatPromptTemplate.from_messages([("system", SYSTEM), ("human", "{question}")])
return prompt | llm | StrOutputParser()
```

In [ ]:
show("TODO 2")
mod = load_student(WORK_REPO)
try:
    chain = mod.make_chain(mod.make_llm())
    print()
    print(chain.invoke({"question": "What are your branch timings on Saturday?"}))
except NotImplementedError as e:
    print("not done yet:", e)

**What you should see.** `PASS TODO 2` and an answer saying 10am to 1pm on Saturday — the model only knows that
because `SYSTEM` reached it through your template.

**Check yourself.** What does `invoke` take here — a string or a dict? (A dict: `{"question": ...}`, because the
template has a `{question}` blank.) *Common mistake:* writing `"{Question}"` in the template and `"question"` in the
dict — names must match exactly.

## Section 4 · TODO 3 — a tool, as a normal Python function
**What and why.** In Session 20 each tool needed about 15 lines of hand-written JSON schema. With `@tool`, LangChain
writes that schema from your function: type hints become the types, and the docstring becomes the description.

**The idea — the most important line today:** **the docstring is now your prompt.** The model never sees your code. It
sees the name, the docstring and the types. A lazy docstring means wrong tool calls. That's Session 20's "the
description is a prompt", in a new place.

`parse_docstring=True` matters. Without it, the `Args:` lines get lumped into one big description. With it, each
argument gets its own description in the schema, so `"e.g. 20 years = 240"` sits right next to `tenure_months` —
exactly where the model needs it.

**What to type** — the docstring is already written for you. Replace the `raise` line in `calculate_emi` with:
```python
r = annual_rate / 12 / 100
return round(principal * r * (1 + r) ** tenure_months / ((1 + r) ** tenure_months - 1))
```
The next cell runs your tool, then prints the schema LangChain generated **next to the one you hand-wrote in Session 20**.

In [ ]:
show("TODO 3")
mod = load_student(WORK_REPO)
from langchain_core.utils.function_calling import convert_to_openai_tool
from tool_tasks import TOOLS as S20_TOOLS
try:
    print()
    print("your tool says:", mod.calculate_emi.invoke({"principal": 2500000, "annual_rate": 8.5, "tenure_months": 240}))
    import json
    mine = convert_to_openai_tool(mod.calculate_emi)["function"]
    print()
    print("GENERATED by LangChain from your function:")
    print(json.dumps(mine["parameters"]["properties"], indent=1))
    print()
    print("HAND-WRITTEN in Session 20:")
    print(json.dumps(S20_TOOLS[0]["function"]["parameters"]["properties"], indent=1))
except NotImplementedError as e:
    print("not done yet:", e)

**What you should see.** `your tool says: 21696`, then two schemas that are almost identical: same three arguments,
same types, same `"e.g. 20 years = 240"`. One was generated from five lines of Python; the other you wrote by hand.

**Try this (2 minutes).** In your file, change `@tool(parse_docstring=True)` to just `@tool`, save, and re-run the
cell. The per-argument descriptions disappear — the checker fails too. Put it back.

**Check yourself.** Where does the model's knowledge of "months, not years" come from? (Only from the docstring.
Delete `20 years = 240` and the model is left guessing — Session 20's stretch demo.)

## Section 5 · TODO 4 — bind the tools, write YOUR capped loop
**What and why.** This is Session 20's round trip, written with LangChain's pieces. Three S20 chores disappear:
- no hand-written schema (`bind_tools` sends the one `@tool` generated)
- no `json.loads` (`call["args"]` is already a dict)
- no hand-made `tool_call_id` (`TOOLS[name].invoke(call)` returns a `ToolMessage` with the right id)

**What doesn't disappear: the loop, and its cap.** You write those. LangChain has a prebuilt agent that hides even the
loop — that's Module 6, and you'll understand it because you wrote this first.

**The idea.**
```
 messages = [SystemMessage, HumanMessage]
 repeat at most max_steps times:
     ai = llm_with_tools.invoke(messages)   ──► append ai
     no tool_calls?  → return ai.content     (the model answered in words)
     else for each call → append TOOLS[name].invoke(call)   (a ToolMessage, same id)
 cap reached → return "(stopped ...)"
```

**What to type** — replace the `raise` line in `ask(...)` with the code in its docstring (also below). Indentation
matters in Python: everything inside `for` is indented 4 more spaces.
```python
messages = [SystemMessage(SYSTEM), HumanMessage(question)]
for step in range(1, max_steps + 1):
    ai = llm_with_tools.invoke(messages)
    messages.append(ai)
    if not ai.tool_calls:
        return ai.content
    for call in ai.tool_calls:
        print(f"  step {step}: {call['name']}({call['args']})")
        messages.append(TOOLS[call["name"]].invoke(call))
return "(stopped: step cap reached)"
```

In [ ]:
show("TODO 4")
mod = load_student(WORK_REPO)
try:
    llm_with_tools = mod.make_llm().bind_tools(list(mod.TOOLS.values()))
    print()
    print(mod.ask(llm_with_tools, "What is the EMI on a 25 lakh home loan at 8.5% for 20 years?"))
except NotImplementedError as e:
    print("not done yet:", e)

**What you should see.** Two `PASS TODO 4` lines (the checker tested your loop against a scripted model, including
one that never stops asking for tools — your cap must stop it at 3). Then a real run:
`step 1: calculate_emi({'principal': 2500000, 'annual_rate': 8.5, 'tenure_months': 240})` and an answer of about
**₹21,696**.

**If yours is different.** No `step 1:` line means the model answered in words without the tool — run again; small
models do this sometimes. `AttributeError: 'str' object has no attribute 'tool_calls'` means you called the chain
instead of `llm_with_tools`. More in `s21_common_errors.md`.

**Check yourself.** Why must `messages.append(ai)` come before the tool results? (The tool result answers the
request inside `ai`; without it, the API rejects the list — S20's most common bug.)

## Section 6 · TODO 5 (stretch) — a second tool, two steps in order
**What and why.** "EMI on 30 lakh at your floating rate over 15 years?" needs today's rate first, then the EMI. With a
`get_rate` tool, your loop runs two tool steps before the model answers.

**What to type** — below the `# TODO 5` comment in your file:
```python
@tool(parse_docstring=True)
def get_rate(product: str, rate_type: str) -> float:
    """Today's interest rate in percent per year. Use when the customer asks for a rate or needs one for an EMI.

    Args:
        product: "home_loan" or "personal_loan"
        rate_type: "floating" or "fixed" (personal loans are fixed only)
    """
    card = {("home_loan", "floating"): 8.5, ("home_loan", "fixed"): 9.0, ("personal_loan", "fixed"): 10.5}
    return card[(product, rate_type)]


TOOLS["get_rate"] = get_rate
```
Write the docstring yourself if you can — it's your prompt.

In [ ]:
mod = load_student(WORK_REPO)
if "get_rate" not in mod.TOOLS:
    print("TODO 5 not done yet (optional)")
else:
    llm_with_tools = mod.make_llm().bind_tools(list(mod.TOOLS.values()))
    print(mod.ask(llm_with_tools, "What would my EMI be on a 30 lakh home loan at your floating rate, over 15 years?"))

**What you should see.** `get_rate(...)` then `calculate_emi(3000000, 8.5, 180)`, and about **₹29,542**. A small model
may skip `get_rate` and guess, or hit the cap — the same behaviours you measured in Session 20.

## Section 7 · Break it on purpose: what `@tool` checks, and what it doesn't
**What and why.** Your tool has type hints, so LangChain validates arguments before your function runs. How much does
that protect you? Session 20's laptop model sent `"0"` strings. Let's see what happens to bad arguments now.

**What the next cell does.** Sends three bad argument sets straight to your tool and prints what happens.

In [ ]:
mod = load_student(WORK_REPO)
for bad in ({"principal": "abc", "annual_rate": 8.5, "tenure_months": 240},
            {"principal": "0", "annual_rate": "0", "tenure_months": "0"},
            {"principal": 2500000, "annual_rate": 0.085, "tenure_months": 240}):
    try:
        print(bad, "->", mod.calculate_emi.invoke(bad))
    except Exception as e:
        print(bad, "->", type(e).__name__)

**What you should see.**

| Arguments | Result | Lesson |
|---|---|---|
| `"abc"` | `ValidationError` | type hints caught it — part of S20's Fix B, for free |
| `"0"`, `"0"`, `"0"` | `ZeroDivisionError` | `"0"` was turned into `0` and accepted — types aren't ranges |
| `annual_rate: 0.085` | **10506** — silently wrong | a valid number with the wrong meaning passes everything |

**Check yourself.** If a bad call happens inside `ask()`, what happens to the whole conversation? (It crashes. The
framework doesn't protect your loop — that's homework: catch the error and send it back as a `ToolMessage`, S20's
Fix B in LangChain.)

## Section 8 · Checkpoint
**What and why.** `check_s21_work` imports **your** file and tests your four functions against scripted stand-ins — no
model call, no cost. It's the same check the pytest checkpoint runs.

In [ ]:
from prompt_utils import results_table
print(results_table(check_s21_work(WORK_REPO)))

## Section 9 · Commit and push your code
Every line `PASS` (TODO 5 says "not attempted - optional" if you skipped it — that's still a pass). Then, in a terminal:
```
cd $env:WORK_REPO
git add s21/bankassist_lc.py
git commit -m "Session 21: BankAssist built with LangChain"
git push
cd <your course repo folder>
uv run pytest tests/test_s21.py -rs
```
**This is the first code on your GitHub that you wrote yourself.** Your counsellors will point recruiters at it.

---
# Part C · Wrap-up

## C1. Summary
- A **framework** writes the repeated parts; it doesn't add powers. BankAssist does exactly what it did in Session 20.
- `ChatOpenAI` is the model as an object; `prompt | llm | parser` is a chain; `@tool` turns a function into a tool.
- **The docstring is your prompt.** Type hints become the schema; `parse_docstring=True` keeps per-argument descriptions.
- `bind_tools` + `tool_calls` + `tool.invoke(call)` remove S20's schema, `json.loads` and id-matching chores.
- **You still own the loop and its cap**, and validation of ranges and meaning.
- Pin versions, and give your AI assistant the version card.

## C2. Honest limits
- **Hidden means harder to debug.** When a framework call misbehaves, you need S20's knowledge to know what it did.
- **Type checks aren't validation.** `@tool` caught `"abc"` but accepted `"0"` and `0.085`.
- **Version churn is real.** LangChain changes its APIs often; today's code is pinned to `langchain==1.4.7`. Code
  from a 2023 tutorial — or from ChatGPT without the card — often won't run.
- **A framework is a dependency.** In a bank, every library is reviewed (licence, maintenance, security). Bigger
  frameworks mean bigger reviews; some teams stay "thin" for that reason.

## C3. Interview questions, answered
1. **What does LangChain add over calling the API directly?** A common interface across providers, message and prompt
   objects, `@tool` schemas from functions, chains with `|`, and ready integrations. It removes plumbing, not decisions.
2. **How does `@tool` know what the tool does?** From the function name, type hints and docstring. The docstring is
   the description the model reads — write it like a prompt.
3. **What happens between `bind_tools` and the final answer?** The model returns `tool_calls`; you run each tool
   (`tool.invoke(call)` gives a `ToolMessage` with the same id), append, and call again until it answers in words.
4. **When would you NOT use LangChain?** A single simple call path, a need for full control and easy debugging, or no
   capacity to track version changes. Prototype raw to learn the protocol; adopt the framework when integrations pay off.
5. **How do you prove a framework migration didn't change behaviour?** Run the same golden set before and after
   (Session 20's ten questions) and compare scores and tokens.

## C4. Common mistakes
See `module03/s21_common_errors.md` for the full table. The top three: running from the wrong folder (`KeyError:
'MODEL'`), calling the chain instead of `llm_with_tools` in `ask()`, and pasting old LangChain code from ChatGPT.

## C5. Homework (choose at least two)
1. **TODO 5**, if you skipped it.
2. **Fix B in LangChain:** in `ask()`, wrap `TOOLS[call["name"]].invoke(call)` in `try/except`. On any error, append
   `ToolMessage(content=f"INVALID: {e}. Answer without a tool if none is needed.", tool_call_id=call["id"])`. Also
   handle a tool name that isn't in `TOOLS`.
3. **Fix C in LangChain:** a router with `llm.with_structured_output(...)` and a small Pydantic model
   (`needs_tool: bool`). No string matching, so the `NO_TOOL` bug can't happen.
4. **Parity:** run your `ask()` on Session 20's ten golden questions (`from tool_tasks import GOLDEN`) and compare
   how many answers contain the right number or fact with your Session 20 report. Same model, same prompts — are the
   results the same? If not, what did the framework change?